# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to explore the FAIR^2 mass spectrometry dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Display dataset metadata (title and description)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's inspect the dataset's record sets and see what tables are present.

In [ ]:
# List all record sets and their IDs
print('Record sets available in dataset:')
pprint.pprint([(rs['@id'], rs.get('name', '')) for rs in dataset._jsonld.get('recordSet', [])])

# For demonstration, gather details for each record set: fields, columns
record_sets_metadata = dataset._jsonld.get('recordSet', [])

for rs in record_sets_metadata:
    print(f"\nRecord Set @id: {rs.get('@id')}")
    print(f"  Name: {rs.get('name', '')}")
    print(f"  Description: {rs.get('description', '')}")
    print('  Fields:')
    for field in rs.get('field', []):
        if isinstance(field, dict):
            print(f"    - @id: {field.get('@id')}")
            print(f"      Name: {field.get('name', '')}")
            print(f"      Data type: {field.get('dataType', '')}")
            print(f"      Description: {field.get('description', '')}")
        else:
            print(f"    - @id: {field}")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s found above.
> Note: The main protein quantification table is typically the largest, and can be selected below. If there is more than one record set, all are loaded.

In [ ]:
# Find all record set @ids
record_set_ids = [rs['@id'] for rs in dataset._jsonld.get('recordSet', [])]
print("Available Record Set @ids:")
for rid in record_set_ids:
    print(f"  {rid}")

# Load each record set as a DataFrame (if available)
dataframes = dict()
for record_set_id in record_set_ids:
    print(f"\nLoading record set {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records. Columns:")
        print(list(df.columns))
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Failed to load record set {record_set_id}: {e}")
        continue

# For demonstration, pick the first record set to explore (often main table)
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"\nFirst record set @id for analysis: {main_rs_id}")
    print(f"Columns: {dataframes[main_rs_id].columns.tolist()}")
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Let's conduct basic statistical analyses:
* Filter for proteins with a high count (or abundance)
* Normalize a numeric field
* Group by categorical fields (e.g. sample or protein type)

You can replace the `numeric_field_id` and `group_field_id` variables with real `@id`s from the dataset for more advanced analysis.

In [ ]:
# Pick a record set and fields (by real or example @ids)
record_set_id = main_rs_id  # e.g. 'cr:RecordSet/Proteins'

# Let's try to detect numeric fields, e.g., searching for 'abundance', 'count', 'coverage', 'MW', etc.
df = dataframes[record_set_id]
possible_numeric_fields = [col for col in df.columns if df[col].dtype in (int, float) or pd.api.types.is_numeric_dtype(df[col])]
print(f"Possible numeric fields: {possible_numeric_fields}")
if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    numeric_field = df.columns[0]  # fallback
print(f"Analyzing numeric field: {numeric_field}")

# Example: filter on proteins with high abundance (or count)
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records where '{numeric_field}' > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the selected numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() + 1e-9)
print(f"Normalized '{numeric_field}' for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try a sample grouping (e.g. by protein description or sample name)
possible_group_fields = [col for col in df.columns if df[col].dtype == object and col != numeric_field]
group_field = possible_group_fields[0] if possible_group_fields else numeric_field
print(f"Grouping by field: {group_field}")
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped and averaged data by '{group_field}':")
    display(grouped_df.head())

## 5. Visualization
Let's plot the distribution of the selected numeric field and show relationships between two fields (if possible).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(df[numeric_field].dropna(), kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If there is a second numeric field, plot scatter
if len(possible_numeric_fields) > 1:
    second_numeric = possible_numeric_fields[1]
    plt.figure(figsize=(7, 5))
    sns.scatterplot(x=df[numeric_field], y=df[second_numeric])
    plt.title(f"{numeric_field} vs {second_numeric}")
    plt.xlabel(numeric_field)
    plt.ylabel(second_numeric)
    plt.show()

# Boxplot by group_field for the main numeric field
if group_field and df[group_field].nunique() < 20:
    plt.figure(figsize=(10, 4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, we have loaded, explored, and visualized the FAIR^2 mass spectrometry dataset using `mlcroissant`.

- We programmatically explored available record sets and fields by their `@id`s and examined dataset structure.
- We loaded the main data table into a DataFrame and conducted exploratory analysis, including filtering, normalization, and aggregation.
- We visualized numeric data distributions and relationships, which can support further biological or statistical analyses.

For more advanced usage, consult documentation for the `mlcroissant` library or further extend this notebook with deeper biological analysis, machine learning pipelines, or integration with curated protein databases.